# EDA: raw pulled data

Loads the raw JSON from `data/raw/` (Open Targets + ClinicalTrials.gov) into DataFrames for exploration. No analysis here — just loading.

In [5]:
import json

import pandas as pd

from pharma_graphrag.config import DATA_RAW_DIR

## Open Targets: targets and drugs

In [6]:
target_rows = []
drug_rows = []

for path in sorted(DATA_RAW_DIR.glob("open_targets_*.json")):
    disease = json.loads(path.read_text())["data"]["disease"]
    disease_name = disease["name"]
    disease_id = disease["id"]

    for row in disease["associatedTargets"]["rows"]:
        target_rows.append(
            {
                "disease_name": disease_name,
                "disease_id": disease_id,
                "score": row["score"],
                "target_id": row["target"]["id"],
                "target_symbol": row["target"]["approvedSymbol"],
            }
        )

    for row in disease["drugAndClinicalCandidates"]["rows"]:
        drug_rows.append(
            {
                "disease_name": disease_name,
                "disease_id": disease_id,
                "drug_id": row["drug"]["id"],
                "drug_name": row["drug"]["name"],
                "drug_type": row["drug"]["drugType"],
                "max_clinical_stage": row["maxClinicalStage"],
            }
        )

targets_df = pd.DataFrame(target_rows)
drugs_df = pd.DataFrame(drug_rows)
targets_df

,disease_name,disease_id,score,target_id,target_symbol
0,Crohn disease,MONDO_0005011,0.772886,ENSG00000167207,NOD2
1,Crohn disease,MONDO_0005011,0.744043,ENSG00000113302,IL12B
2,Crohn disease,MONDO_0005011,0.723476,ENSG00000115232,ITGA4
3,Crohn disease,MONDO_0005011,0.690574,ENSG00000105397,TYK2
4,Crohn disease,MONDO_0005011,0.686728,ENSG00000096968,JAK2
...,...,...,...,...,...
245,rheumatoid arthritis,MONDO_0008383,0.548671,ENSG00000107485,GATA3
246,rheumatoid arthritis,MONDO_0008383,0.545644,ENSG00000213626,LBH
247,rheumatoid arthritis,MONDO_0008383,0.534106,ENSG00000136573,BLK
248,rheumatoid arthritis,MONDO_0008383,0.530848,ENSG00000150347,ARID5B


In [7]:
drugs_df

,disease_name,disease_id,drug_id,drug_name,drug_type,max_clinical_stage
0,Crohn disease,MONDO_0005011,CHEMBL5314733,DUVAKITUG,Antibody,PHASE_3
1,Crohn disease,MONDO_0005011,CHEMBL1535,HYDROXYCHLOROQUINE,Small molecule,PHASE_2
2,Crohn disease,MONDO_0005011,CHEMBL4297477,BREPOCITINIB,Small molecule,PHASE_2
3,Crohn disease,MONDO_0005011,CHEMBL2104987,TEDUGLUTIDE,Protein,PHASE_2
4,Crohn disease,MONDO_0005011,CHEMBL1201821,PALIFERMIN,Protein,PHASE_1_2
...,...,...,...,...,...,...
1197,rheumatoid arthritis,MONDO_0008383,CHEMBL1742483,RESOLVIN E1,Small molecule,PHASE_1
1198,rheumatoid arthritis,MONDO_0008383,CHEMBL5095370,TULISOKIBART,Antibody,PHASE_2
1199,rheumatoid arthritis,MONDO_0008383,CHEMBL454950,TETRACYCLINE HYDROCHLORIDE,Small molecule,PHASE_2
1200,rheumatoid arthritis,MONDO_0008383,CHEMBL1908355,TENIDAP,Small molecule,PHASE_3


## ClinicalTrials.gov: studies

In [9]:
trial_rows = []

for path in sorted(DATA_RAW_DIR.glob("clinical_trials_*.json")):
    query_bucket = path.stem.removeprefix("clinical_trials_")
    studies = json.loads(path.read_text())["studies"]

    for study in studies:
        protocol = study["protocolSection"]
        trial_rows.append(
            {
                "query_bucket": query_bucket,
                "nct_id": protocol["identificationModule"].get("nctId"),
                "conditions": protocol.get("conditionsModule", {}).get(
                    "conditions", []
                ),
                "keywords": protocol.get("conditionsModule", {}).get("keywords", []),
                "brief_summary": protocol.get("descriptionModule", {}).get(
                    "briefSummary"
                ),
                "phases": protocol.get("designModule", {}).get("phases", []),
                "overall_status": protocol.get("statusModule", {}).get("overallStatus"),
                "lead_sponsor": protocol.get("sponsorCollaboratorsModule", {})
                .get("leadSponsor", {})
                .get("name"),
            }
        )

trials_df = pd.DataFrame(trial_rows)
trials_df

,query_bucket,nct_id,conditions,keywords,brief_summary,phases,overall_status,lead_sponsor
0,crohns_disease,NCT06581042,"[Ulcerative Colitis, Crohn's Disease]","[Ulcerative colitis, Crohn's disease, Upadacit...",Crohn's disease (CD) is an incurable chronic i...,[],RECRUITING,AbbVie
1,crohns_disease,NCT07077746,[Crohn Disease (CD)],"[Crohn&#39;s, Crohn&#39;s Disease, Autoimmune,...","Methodology: Randomized, double-blind, efficac...",[PHASE2],RECRUITING,Hope Biosciences Research Foundation
2,crohns_disease,NCT01199302,[Crohn's Disease],"[Amgen, Crohn's, Inflammatory Bowel Disease, I...",The purpose of this study is to evaluate the s...,[PHASE2],TERMINATED,Amgen
3,crohns_disease,NCT02704624,"[Crohn Disease, Vitamin D Deficiency, Fatigue,...","[Crohn Disease, Fatigue, Osteopenia, Exercise ...","Thus, the aim of the current study is to asses...",[PHASE4],UNKNOWN,Federal University of Juiz de Fora
4,crohns_disease,NCT01559142,[Crohn Disease],"[Crohn Disease, infliximab, azathioprine]",The aim of the study is confirmation of effica...,[PHASE3],UNKNOWN,"Children's Memorial Health Institute, Poland"
...,...,...,...,...,...,...,...,...
495,rheumatoid_arthritis,NCT00632710,[Rheumatoid Arthritis],"[rheumatoid arthritis, hands, randomized contr...",The purpose of the present study was to assess...,[NA],COMPLETED,Federal University of São Paulo
496,rheumatoid_arthritis,NCT00096759,"[Arthritis, Rheumatoid]","[Arthritis, Tai Chi, Meditation, Relaxation Te...",The purpose of this study is to compare the ef...,"[PHASE2, PHASE3]",COMPLETED,National Center for Complementary and Integrat...
497,rheumatoid_arthritis,NCT00502424,[Rheumatoid Arthritis.],"[Rheumatoid Arthritis, Hand, Splint, Occupatio...",The purpose of the present study was to evalua...,[PHASE2],COMPLETED,Federal University of São Paulo
498,rheumatoid_arthritis,NCT02832180,[Arthritis],[Rheumatoid],"This is an open-label, 2-cycle, multiple-dose,...",[PHASE1],COMPLETED,Bristol-Myers Squibb
